In [1]:
print("Welcome to Spam Email Classifier!")
print("My first Machine Learning Project")


Welcome to Spam Email Classifier!
My first Machine Learning Project


In [3]:
import pandas as pd
import sklearn
import joblib

print("All libraries are working!")

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Joblib version:", joblib.__version__)

All libraries are working!
Pandas version: 2.2.3
Scikit-learn version: 1.6.1
Joblib version: 1.6.0


In [4]:
import urllib.request

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

urllib.request.urlretrieve(url, "spam_dataset.zip")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [5]:
import zipfile

with zipfile.ZipFile("spam_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("spam_data")

print("Dataset extracted successfully!")


Dataset extracted successfully!


In [6]:
import urllib.request

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

urllib.request.urlretrieve(url, "spam_dataset.zip")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [7]:
import zipfile

with zipfile.ZipFile("spam_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("spam_data")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [8]:
import pandas as pd

file_path = "spam_data/SMSSpamCollection"

data = pd.read_csv(
    file_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")
print("Total messages:", len(data))

data.head()

Dataset loaded successfully!
Total messages: 5572


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
print("Dataset information:")
print(data.info())

print("\nSpam and Ham counts:")
print(data["label"].value_counts())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None

Spam and Ham counts:
label
ham     4825
spam     747
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

# Input messages
X = data["message"]

# Correct labels
y = data["label"]

# Divide dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

print("Data splitting completed successfully!")

Training messages: 4457
Testing messages: 1115
Data splitting completed successfully!


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text messages into numerical features
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print("Text conversion completed successfully!")

print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Text conversion completed successfully!
Training data shape: (4457, 7403)
Testing data shape: (1115, 7403)


In [12]:
from sklearn.naive_bayes import MultinomialNB

# Create the machine learning model
model = MultinomialNB()

# Train the model using our training data
model.fit(X_train_tfidf, y_train)

print("Machine Learning Model Trained Successfully!")

Machine Learning Model Trained Successfully!


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict labels for testing messages
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("MODEL EVALUATION")
print("-------------------------")
print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

MODEL EVALUATION
-------------------------
Model Accuracy: 97.04 %

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.78      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[966   0]
 [ 33 116]]


In [14]:
def check_message(message):

    # Convert message into numerical format
    message_vector = vectorizer.transform([message])

    # Predict the message category
    prediction = model.predict(message_vector)[0]

    if prediction == "spam":
        print("🚨 WARNING: This message is SPAM!")
    else:
        print("✅ This message is NOT SPAM.")

    print("Your message:", message)


# Test message 1
check_message("Congratulations! You won a free prize. Claim now!")

print("\n-------------------------\n")

# Test message 2
check_message("Hello, how are you? See you tomorrow.")

🚨 WARNING: This message is SPAM!
Your message: Congratulations! You won a free prize. Claim now!

-------------------------

✅ This message is NOT SPAM.
Your message: Hello, how are you? See you tomorrow.


In [15]:
check_message("URGENT! You have won a lottery. Call now!")

🚨 WARNING: This message is SPAM!
Your message: URGENT! You have won a lottery. Call now!


In [16]:
check_message("Please send me the project report tomorrow.")

✅ This message is NOT SPAM.
Your message: Please send me the project report tomorrow.


In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create application heading
heading = widgets.HTML(
    value="<h2>📧 Spam Message Detector</h2>"
          "<p>Enter a message below to check whether it is spam.</p>"
)

# Create message input box
message_input = widgets.Textarea(
    placeholder="Type your message here...",
    description="Message:",
    layout=widgets.Layout(width="90%", height="100px")
)

# Create button
check_button = widgets.Button(
    description="Check Message",
    button_style="primary",
    icon="search"
)

# Create output area
result_output = widgets.Output()

# Prediction function
def predict_message(button):
    with result_output:
        clear_output()

        message = message_input.value.strip()

        if not message:
            print("Please enter a message!")
            return

        message_vector = vectorizer.transform([message])
        prediction = model.predict(message_vector)[0]

        if prediction == "spam":
            print("🚨 WARNING: THIS MESSAGE IS SPAM!")
        else:
            print("✅ THIS MESSAGE IS NOT SPAM!")

        print("\nMessage:", message)

# Connect button to function
check_button.on_click(predict_message)

# Display application
display(heading, message_input, check_button, result_output)

HTML(value='<h2>📧 Spam Message Detector</h2><p>Enter a message below to check whether it is spam.</p>')

Textarea(value='', description='Message:', layout=Layout(height='100px', width='90%'), placeholder='Type your …

Button(button_style='primary', description='Check Message', icon='search', style=ButtonStyle())

Output()

In [18]:
import joblib

# Save trained model and text vectorizer
joblib.dump(model, "spam_classifier_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully!")
print("Vectorizer saved successfully!")

Model saved successfully!
Vectorizer saved successfully!


In [20]:
from google.colab import files

files.download("spam_classifier_model.pkl")
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>